# Notebook 02 — Extração de Features

**Projeto Final de Visão Computacional — Cenário C: Inspeção de Comprimidos**

Este notebook cumpre o **Bloco 3 da rubrica (1,0 ponto)**: transformar cada comprimido
segmentado em um **vetor de features manuais**, cobrindo as **quatro famílias obrigatórias**
do enunciado:

| Família | Features |
|---|---|
| **Forma** | area, perimeter, circularity, solidity, extent, eccentricity |
| **Inerciais** | 7 momentos de Hu (escala log) |
| **Cor** | média H, S, V e média B, G, R dentro da máscara |
| **Textura** | 4 propriedades de GLCM (contrast, homogeneity, energy, correlation) |

**Saída:** os arquivos `X.csv` (features) e `y.csv` (classes), exigidos no repositório.

## 1. Configuração

Mesmos parâmetros do Notebook 01. A coluna `split` preserva a divisão train/valid/test que já vem pronta no OGYEIv2 — útil no Notebook 03.

In [9]:
import os, sys
sys.path.append(os.path.abspath(".."))

import os, glob, re
from pathlib import Path
from collections import Counter
import cv2
import numpy as np
import pandas as pd
from skimage.feature import graycomatrix, graycoprops

# caminho do dataset (ver config.py)
from config import DATASET_ROOT
# classes do projeto
CLASSES = [
    "algoflex_forte_dolo_400_mg",
    "apranax_550_mg",
    "donalgin_250_mg",
    "jutavit_c_vitamin"
]

os.makedirs("outputs", exist_ok=True)
np.random.seed(42)

## 2. Funções de leitura e segmentação

Reaproveitamos a segmentação escolhida no Notebook 01. Aqui usamos Otsu como padrão — troque por `segmentar_hsv` se a equipe preferir.

In [10]:
def classe_do_arquivo(caminho):
    return re.sub(r'_[su]_\d+$', '', Path(caminho).stem)

def listar_imagens(split):
    padrao = os.path.join(DATASET_ROOT, split, "images", "*.jpg")
    return [p for p in glob.glob(padrao) if classe_do_arquivo(p) in CLASSES]

def redimensionar(bgr, max_dim=600):
    h, w = bgr.shape[:2]
    escala = max_dim / max(h, w)
    if escala < 1:
        bgr = cv2.resize(bgr, (int(w*escala), int(h*escala)), interpolation=cv2.INTER_AREA)
    return bgr

def limpar_mascara(mask):
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, k)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k)
    cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    out = np.zeros_like(mask)
    if cnts:
        cv2.drawContours(out, [max(cnts, key=cv2.contourArea)], -1, 255, cv2.FILLED)
    return out

def segmentar_saturacao(bgr):
    """segmenta pelo canal de saturação"""
    bgr = redimensionar(bgr)
    s = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)[:, :, 1]
    _, th = cv2.threshold(s, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return limpar_mascara(th)

def segmentar_otsu(bgr):
    """Otsu com inversao de polaridade automatica"""
    bgr = redimensionar(bgr)
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    _, th = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    if (th > 0).sum() > (th == 0).sum():
        th = cv2.bitwise_not(th)
    return limpar_mascara(th)

SEGMENTAR = segmentar_otsu   # método escolhido (Otsu funcionou melhor nas imagens reais)


## 3. Extração das quatro famílias de features

Cada função abaixo calcula uma família. A máscara garante que cor e textura sejam medidas
**apenas dentro do comprimido**, não no fundo.

In [11]:
def features_forma(contorno):
    f = {}
    area = cv2.contourArea(contorno)
    per  = cv2.arcLength(contorno, True)
    f['area'] = area
    f['perimeter'] = per
    f['circularity'] = (4*np.pi*area/(per*per)) if per > 0 else 0.0
    hull = cv2.convexHull(contorno); ha = cv2.contourArea(hull)
    f['solidity'] = area/ha if ha > 0 else 0.0
    x, y, w, h = cv2.boundingRect(contorno)
    f['extent'] = area/(w*h) if w*h > 0 else 0.0
    if len(contorno) >= 5:
        (_, _), (MA, ma), _ = cv2.fitEllipse(contorno)
        f['eccentricity'] = np.sqrt(1-(min(MA,ma)/max(MA,ma))**2) if max(MA,ma) > 0 else 0.0
    else:
        f['eccentricity'] = 0.0
    return f

def features_hu(mask):
    hu = cv2.HuMoments(cv2.moments(mask)).flatten()
    # Hu em escala log
    return {f'hu{i+1}': float(-np.sign(v)*np.log10(abs(v)+1e-30)) for i, v in enumerate(hu)}

def features_cor(bgr, mask):
    f = {}
    hsv = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)
    m = mask > 0
    for i, ch in enumerate(['h','s','v']): f[f'mean_{ch}'] = float(hsv[:,:,i][m].mean())
    for i, ch in enumerate(['b','g','r']): f[f'mean_{ch}'] = float(bgr[:,:,i][m].mean())
    return f

def features_textura(bgr, mask):
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY).copy()
    gray[mask == 0] = 0
    # 8 níveis p/ GLCM
    glcm = graycomatrix((gray//32).astype(np.uint8), distances=[1], angles=[0],
                        levels=8, symmetric=True, normed=True)
    return {f'glcm_{p}': float(graycoprops(glcm, p)[0,0])
            for p in ['contrast','homogeneity','energy','correlation']}

def extrair_features(bgr, mask):
    cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts:
        return None
    c = max(cnts, key=cv2.contourArea)
    if cv2.contourArea(c) < 50:   # máscara degenerada
        return None
    f = {}
    f.update(features_forma(c))
    f.update(features_hu(mask))
    f.update(features_cor(bgr, mask))
    f.update(features_textura(bgr, mask))
    return f


## 4. Montagem da tabela X e do vetor y

Percorremos os três splits, preservando a divisão original na coluna `split`.

In [12]:
linhas, classes = [], []
falhas = 0
for split in ["train", "valid", "test"]:
    for p in listar_imagens(split):
        bgr = cv2.imread(p)
        if bgr is None:
            falhas += 1; continue
        bgr = redimensionar(bgr)
        mask = SEGMENTAR(bgr)
        feats = extrair_features(bgr, mask)
        if feats is None:
            falhas += 1; continue
        feats['split'] = split
        linhas.append(feats)
        classes.append(classe_do_arquivo(p))

X = pd.DataFrame(linhas)
y = pd.Series(classes, name="classe")
print("X shape:", X.shape, "| falhas de segmentação:", falhas)
print("Distribuição por classe:", y.value_counts().to_dict())
print("Distribuição por split :", X['split'].value_counts().to_dict())
print("NaN totais:", int(X.isna().sum().sum()))
X.head()

X shape: (160, 24) | falhas de segmentação: 0
Distribuição por classe: {'algoflex_forte_dolo_400_mg': 40, 'apranax_550_mg': 40, 'donalgin_250_mg': 40, 'jutavit_c_vitamin': 40}
Distribuição por split : {'train': 112, 'valid': 24, 'test': 24}
NaN totais: 0


,area,perimeter,circularity,solidity,extent,eccentricity,hu1,hu2,hu3,hu4,...,mean_s,mean_v,mean_b,mean_g,mean_r,glcm_contrast,glcm_homogeneity,glcm_energy,glcm_correlation,split
0,82316.5,2543.325024,0.159916,0.525420,0.424750,0.898971,2.712383,5.613316,8.947972,9.847518,...,10.429608,162.329629,161.470927,156.364827,159.308514,0.121167,0.982236,0.636158,0.987692,train
1,83800.0,2304.430582,0.198302,0.535226,0.432405,0.886637,2.722128,5.634155,8.974292,9.878867,...,9.494822,161.677934,160.838237,156.571191,158.631773,0.105077,0.982886,0.632695,0.989350,train
2,87203.5,2407.241764,0.189106,0.557776,0.449966,0.883896,2.737799,5.671060,9.168828,9.937635,...,13.158611,162.862890,159.865799,154.514668,159.948100,0.112442,0.981556,0.620543,0.988610,train
3,77760.0,2616.621487,0.142720,0.503394,0.401238,0.890550,2.682031,5.548610,8.834586,9.724539,...,13.382717,161.645827,158.180734,153.189763,158.776919,0.130160,0.982212,0.651645,0.985959,train
4,79342.5,2491.124014,0.160666,0.512901,0.409404,0.905675,2.693849,5.573260,8.914929,9.787324,...,13.383246,161.867903,158.494993,153.368798,158.992881,0.123059,0.980873,0.646641,0.986807,train


## 5. Salvar X.csv e y.csv

Estes dois arquivos são **exigidos no repositório** (Bloco 6 da rubrica) e são a entrada
do Notebook 03 (classificação).

In [15]:
import os
raiz = os.path.abspath("..")   # sobe da pasta notebooks para a raiz do projeto
X.to_csv(os.path.join(raiz, "X.csv"), index=False)
y.to_csv(os.path.join(raiz, "y.csv"), index=False)
print("X.csv e y.csv salvos na raiz do projeto.")
print("Colunas de X:", list(X.columns))

X.csv e y.csv salvos na raiz do projeto.
Colunas de X: ['area', 'perimeter', 'circularity', 'solidity', 'extent', 'eccentricity', 'hu1', 'hu2', 'hu3', 'hu4', 'hu5', 'hu6', 'hu7', 'mean_h', 'mean_s', 'mean_v', 'mean_b', 'mean_g', 'mean_r', 'glcm_contrast', 'glcm_homogeneity', 'glcm_energy', 'glcm_correlation', 'split']


## 6. Conferência rápida das features por classe

Uma prévia das médias por classe ajuda a antecipar a análise do Notebook 03 e já indica
se as features escolhidas separam as classes (ex.: `mean_h`/`mean_s` devem diferir bastante
entre um comprimido branco e uma cápsula colorida).

In [16]:
cols_amostra = ['area','circularity','solidity','mean_h','mean_s','mean_v','glcm_contrast']
X_tmp = X.copy(); X_tmp['classe'] = y.values
X_tmp.groupby('classe')[cols_amostra].mean().round(2)

,area,circularity,solidity,mean_h,mean_s,mean_v,glcm_contrast
classe,,,,,,,
algoflex_forte_dolo_400_mg,41782.61,0.48,0.75,142.58,71.28,197.50,0.07
apranax_550_mg,35427.59,0.45,0.73,106.95,79.63,172.22,0.07
donalgin_250_mg,34675.25,0.47,0.77,121.79,115.89,194.57,0.05
jutavit_c_vitamin,34690.55,0.48,0.78,65.46,75.69,183.25,0.05
